# Step 1 — Entity NER labeling via local LLM (MLX + Qwen2.5-7B-Instruct)

No entity-labeled dataset exists yet. Dish/drink names in `data/normalized/intent_dataset_normalized.jsonl` go beyond `data/menu.json`'s 11 items (e.g. "nasi kuning", "pasta", "ikan goreng" have no menu match), so a rule-based span matcher would under-label badly. Instead, a local LLM (MLX, `mlx-community/Qwen2.5-7B-Instruct-4bit`) labels entity spans per utterance against the fixed schema below, which are then converted to BIO tags.

In [1]:
import json
import random
from pathlib import Path

from mlx_lm import load, generate

## Step 2 — Entity schema and prompt

In [2]:
ENTITY_TAGS = [
    "DISH", "QUANTITY", "MODIFIER", "SIZE", "DRINK",
    "ADD_ON", "REMOVE", "TABLE", "TAKE_AWAY",
]

SYSTEM_PROMPT = """You are an NER labeling assistant for a restaurant ordering system in Surabaya, Indonesia (Indonesian/Javanese-accented speech, code-mixed with English).

Entity tags:
- DISH: food item name (e.g. nasi goreng, grilled chicken, ayam bakar)
- QUANTITY: amount ordered (e.g. dua porsi, satu, three)
- MODIFIER: spice/taste/style modifier (e.g. pedas banget, extra cheese, tanpa bawang)
- SIZE: portion size (e.g. large, ukuran kecil)
- DRINK: beverage name (e.g. es teh, orange juice)
- ADD_ON: additional item added to order (e.g. pakai telur, add sambal)
- REMOVE: item explicitly excluded (e.g. jangan pakai, no onion)
- TABLE: table reference (e.g. meja tiga, table five)
- TAKE_AWAY: takeaway/dine-in indicator (e.g. dibungkus, for takeaway)

Given a sentence, return ONLY a JSON list of entity spans found, each as {"text": <exact substring from sentence>, "label": <TAG>}. Use exact substrings as they appear in the sentence (same casing, same spacing). If no entities found, return []. Do not invent text that is not in the sentence."""

def build_prompt(tokenizer, text: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        add_generation_prompt=True,
        tokenize=False,
    )

## Step 3 — Load model

In [3]:
MODEL_ID = "mlx-community/Qwen2.5-7B-Instruct-4bit"
model, tokenizer = load(MODEL_ID)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

### Sanity check on a few hand-picked sentences

In [4]:
demo_sentences = [
    "tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya",
    "saya mau saya mau memesan pasta dua mangkok",
    "pesenin ikan goreng empat buat meja gue ya pak",
]

for text in demo_sentences:
    prompt = build_prompt(tokenizer, text)
    out = generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=False)
    print(text)
    print("->", out)
    print()


tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya
-> [{"text": "nasi kuning", "label": "DISH"}, {"text": "tiga porsi", "label": "QUANTITY"}, {"text": "dibawa pulah", "label": "TAKE_AWAY"}]

saya mau saya mau memesan pasta dua mangkok
-> [{"text": "pasta", "label": "DISH"}, {"text": "dua mangkok", "label": "QUANTITY"}]

pesenin ikan goreng empat buat meja gue ya pak
-> [{"text": "empat", "label": "QUANTITY"}, {"text": "ikan goreng", "label": "DISH"}, {"text": "meja", "label": "TABLE"}]



## Step 4 — Batch labeling over the full dataset

In [5]:
def parse_entities(raw: str) -> list[dict] | None:
    """
    Parse the LLM's JSON list output. Returns None on parse failure
    (caller logs and retries/skips) rather than silently treating it as [].
    """
    raw = raw.strip()
    start, end = raw.find("["), raw.rfind("]")
    if start == -1 or end == -1:
        return None
    try:
        parsed = json.loads(raw[start : end + 1])
    except json.JSONDecodeError:
        return None
    if not isinstance(parsed, list):
        return None
    for item in parsed:
        if not isinstance(item, dict) or "text" not in item or "label" not in item:
            return None
        if item["label"] not in ENTITY_TAGS:
            return None
    return parsed


def label_sentence(text: str, max_retries: int = 2) -> list[dict] | None:
    prompt = build_prompt(tokenizer, text)
    for _ in range(max_retries):
        raw = generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=False)
        entities = parse_entities(raw)
        if entities is not None:
            return entities
    return None


In [6]:
input_path = Path("../data/normalized/intent_dataset_normalized.jsonl")
records = [json.loads(line) for line in input_path.open(encoding="utf-8")]
print(f"{len(records)} records loaded")

2060 records loaded


In [ ]:
labeled_raw = []
failures = []

for i, record in enumerate(records): 
    text = record["text_normalized"]
    entities = label_sentence(text)
    if entities is None:
        failures.append(record["id"])
        entities = []
    labeled_raw.append({**record, "entities": entities})
    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(records)} labeled, {len(failures)} failures so far")

print(f"done. {len(failures)} failures: {failures}")


100/2060 labeled, 0 failures so far
200/2060 labeled, 0 failures so far
300/2060 labeled, 0 failures so far
400/2060 labeled, 0 failures so far
500/2060 labeled, 0 failures so far
600/2060 labeled, 1 failures so far
700/2060 labeled, 4 failures so far
800/2060 labeled, 4 failures so far
900/2060 labeled, 4 failures so far
1000/2060 labeled, 4 failures so far


### Save raw LLM-labeled output (entities as text spans, pre-BIO)

In [ ]:
raw_output_path = Path("../data/ner_dataset_llm_labeled.jsonl")
with raw_output_path.open("w", encoding="utf-8") as f:
    for rec in labeled_raw:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"wrote {len(labeled_raw)} records to {raw_output_path}")


wrote 1100 records to ../data/ner_dataset_llm_labeled.jsonl


## Step 5 — Span -> BIO conversion

Reuses the token-matching approach from `notebooks/3.2_bio_labeling.ipynb`'s `find_phrase`.

In [ ]:
def find_phrase(tokens: list[str], phrase: str) -> int:
    """
    Return the start index of the first occurrence of `phrase` (space-separated
    tokens) inside `tokens`. Returns -1 if not found. Exact token match after lowercasing.
    """
    phrase_tokens = phrase.lower().split()
    n = len(phrase_tokens)
    if n == 0:
        return -1
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            return i
    return -1


def entities_to_bio(tokens: list[str], entities: list[dict]) -> tuple[list[str], list[str]]:
    """
    Converts LLM entity spans into BIO labels aligned to `tokens`.
    Returns (labels, unmatched_spans) — unmatched spans are logged, not silently dropped,
    since LLM output text doesn't always match tokenization exactly (e.g. punctuation drift).
    """
    labels = ["O"] * len(tokens)
    unmatched = []

    for ent in entities:
        span_tokens = ent["text"].lower().split()
        idx = find_phrase(tokens, ent["text"])
        if idx == -1:
            unmatched.append(ent["text"])
            continue
        n = len(span_tokens)
        if labels[idx] != "O":
            continue  # already labeled by an earlier (assumed higher-priority) span
        labels[idx] = f"B-{ent['label']}"
        for j in range(idx + 1, idx + n):
            labels[j] = f"I-{ent['label']}"

    return labels, unmatched


In [ ]:
def validate_bio(tokens: list[str], labels: list[str], record_id) -> list[str]:
    """
    Auto-corrects orphan I- tags (no preceding B-/I- of the same type) to B-,
    mirroring notebooks/3.2's validate_bio.
    """
    fixed = list(labels)
    prev_type = None
    for i, label in enumerate(fixed):
        if label == "O":
            prev_type = None
            continue
        prefix, _, tag = label.partition("-")
        if prefix == "I" and prev_type != tag:
            fixed[i] = f"B-{tag}"
        prev_type = tag
    return fixed


In [ ]:
bio_records = []
all_unmatched = []

for rec in labeled_raw:
    tokens = rec["text_normalized"].split()
    labels, unmatched = entities_to_bio(tokens, rec["entities"])
    labels = validate_bio(tokens, labels, rec["id"])
    if unmatched:
        all_unmatched.append({"id": rec["id"], "text": rec["text_normalized"], "unmatched": unmatched})
    bio_records.append({
        "id": rec["id"],
        "text": rec["text_normalized"],
        "intent": rec["intent"],
        "domain": rec["domain"],
        "tokens": tokens,
        "labels": labels,
        "source_model": MODEL_ID,
    })

print(f"{len(bio_records)} records converted, {len(all_unmatched)} with unmatched spans")

### Inspect unmatched spans (LLM text didn't align to a token span)

In [ ]:
for item in all_unmatched[:20]:
    print(item)

{'id': 1, 'text': 'tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya', 'unmatched': ['dibawa pulah']}
{'id': 41, 'text': 'kalau boleh saya ingin memesan mie goreng seafood tapi nasinya dipisah ya', 'unmatched': ['nasi']}
{'id': 233, 'text': 'anu gue mau sundae yang pedas sedang ya', 'unmatched': ['sunde']}
{'id': 248, 'text': 'kalau boleh saya ingin memesan ayam geprek tapi nasinya dipisah ya', 'unmatched': ['nasi']}
{'id': 254, 'text': 'minumnya kopi hitam ya ayam goreng satu aja', 'unmatched': ['a']}
{'id': 257, 'text': 'kalau boleh saya ingin memesan ayam penyet tapi nasinya dipisah ya', 'unmatched': ['nasi']}
{'id': 259, 'text': 'tolong nasi uduk eee mie goreng panas ya yang saya mau', 'unmatched': ['nasu uduk']}
{'id': 274, 'text': 'ini pesanan buat dibawa pulang ya bang bungkus aja', 'unmatched': ['dibawa pulah', 'dibungkus']}
{'id': 283, 'text': 'tolong pesankan sup buntut dua porsi untuk dibawa pulang ya', 'unmatched': ['dibawa pulah']}
{'id': 353, 'text': 'ini pesa

## Step 6 — Spot-check before trusting the full labeled set

In [ ]:
def print_spot_check(records: list[dict], n: int = 15, seed: int = 42):
    sample = random.Random(seed).sample(records, k=min(n, len(records)))
    for rec in sample:
        print(rec["text"])
        for tok, lab in zip(rec["tokens"], rec["labels"]):
            if lab != "O":
                print(f"  {tok:20s} {lab}")
        print()

print_spot_check(bio_records)


kak nasinya dipisah ya dari mie ayam nya
  nasinya              B-DISH
  mie                  B-DISH
  ayam                 I-DISH

untuk teman saya eee eh dia juga mau club sandwich sama seperti saya

untuk minumannya thai tea eh wait jus mangga saja
  thai                 B-DRINK
  tea                  I-DRINK
  jus                  B-DRINK
  mangga               I-DRINK

saya mau tambah tambah nasi goreng pete satu porsi lagi
  tambah               B-DISH
  nasi                 I-DISH
  goreng               I-DISH
  pete                 I-DISH
  satu                 B-QUANTITY
  porsi                I-QUANTITY

nasinya minta goreng maaf maksudnya putih ya
  nasinya              B-DISH
  putih                B-MODIFIER

permisi saya ingin memesan spaghetti satu gelas untuk meja tiga
  satu                 B-QUANTITY
  gelas                I-QUANTITY
  meja                 B-TABLE
  tiga                 I-TABLE

sih sebenernya gue mau espresso tapi kalo gak ada steak juga oke

pesanan

In [ ]:
def print_statistics(records: list[dict]):
    from collections import Counter
    tag_counts = Counter()
    for rec in records:
        for lab in rec["labels"]:
            if lab != "O":
                tag_counts[lab.split("-", 1)[1]] += 1
    for tag, count in tag_counts.most_common():
        print(f"{tag:12s} {count}")

print_statistics(bio_records)


DISH         1657
QUANTITY     653
DRINK        513
MODIFIER     229
TABLE        114
REMOVE       75
ADD_ON       58
SIZE         25
TAKE_AWAY    18


### Save BIO-labeled dataset

In [ ]:
bio_output_path = Path("../data/ner_dataset_bio.jsonl")
with bio_output_path.open("w", encoding="utf-8") as f:
    for rec in bio_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"wrote {len(bio_records)} records to {bio_output_path}")


wrote 1100 records to ../data/ner_dataset_bio.jsonl


## Step 7 — Tokenizer alignment (IndoBERT WordPiece)

In [ ]:
from transformers import AutoTokenizer as _AutoTokenizer

INDOBERT_MODEL_NAME = "indobenchmark/indobert-base-p2"
MAX_LENGTH = 64

bert_tokenizer = _AutoTokenizer.from_pretrained(INDOBERT_MODEL_NAME)

all_tags = sorted({lab.split("-", 1)[1] for rec in bio_records for lab in rec["labels"] if lab != "O"})
label_list = ["O"] + [f"{p}-{t}" for t in all_tags for p in ("B", "I")]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

In [ ]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    """
    Tokenize a pre-split word list and align word-level BIO labels to
    subword tokens using word_ids(). Mirrors notebooks/3.2's helper.

    First subword of each word -> label2id[word_label]
    Continuation subwords      -> -100 (ignored by CrossEntropyLoss)
    Special tokens ([CLS]/[SEP]/[PAD]) -> -100
    """
    encoding = bert_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    word_ids = encoding.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            aligned_labels.append(label2id[labels[word_id]])
        else:
            aligned_labels.append(-100)
        prev_word_id = word_id

    encoding["labels"] = aligned_labels
    return encoding


### Align the full dataset

In [ ]:
aligned_records = []
truncated_count = 0

for rec in bio_records:
    encoding = align_labels_with_tokens(rec["tokens"], rec["labels"])
    if len(rec["tokens"]) + 2 > MAX_LENGTH:  # +2 for [CLS]/[SEP]
        truncated_count += 1
    aligned_records.append({
        "id": rec["id"],
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": encoding["labels"],
    })
    assert len(encoding["input_ids"]) == len(encoding["labels"])

print(f"{len(aligned_records)} records aligned, {truncated_count} truncated")


1100 records aligned, 0 truncated


### Save aligned dataset

In [ ]:
aligned_output_path = Path("../data/ner_dataset_aligned.jsonl")
with aligned_output_path.open("w", encoding="utf-8") as f:
    for rec in aligned_records:
        f.write(json.dumps(rec) + "\n")
print(f"wrote {len(aligned_records)} records to {aligned_output_path}")

label_map_path = Path("../data/ner_label_map.json")
label_map_path.write_text(json.dumps({"label2id": label2id, "id2label": id2label}, indent=2))
print(f"wrote label map to {label_map_path}")


wrote 1100 records to ../data/ner_dataset_aligned.jsonl
wrote label map to ../data/ner_label_map.json


## Step 8 — Dataset splitting

No single dominant entity type per row (unlike disfluency's `disfluency_type`), so a plain random split is used instead of stratification.

In [ ]:
from sklearn.model_selection import train_test_split

ids_to_aligned = {rec["id"]: rec for rec in aligned_records}
ids_to_bio = {rec["id"]: rec for rec in bio_records}

all_ids = list(ids_to_aligned.keys())
train_ids, rest_ids = train_test_split(all_ids, test_size=0.2, random_state=42)
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, random_state=42)

print(f"train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}")


train=880 val=110 test=110


### Save splits

In [ ]:
splits = {
    "train": train_ids,
    "val": val_ids,
    "test": test_ids,
}

for split_name, ids in splits.items():
    out_path = Path(f"../data/ner_dataset_{split_name}.jsonl")
    with out_path.open("w", encoding="utf-8") as f:
        for rec_id in ids:
            f.write(json.dumps(ids_to_aligned[rec_id]) + "\n")
    print(f"wrote {len(ids)} records to {out_path}")


wrote 880 records to ../data/ner_dataset_train.jsonl
wrote 110 records to ../data/ner_dataset_val.jsonl
wrote 110 records to ../data/ner_dataset_test.jsonl
